
# Batch Rename Imagery & Labels from CSV

This notebook renames imagery files (and their paired label files) listed in a CSV to a consistent scheme:
- Images → `image_{n}{ext}`
- Labels → `label_{n}{ext}`

Where `n` counts from `START_INDEX` (default `1`) to the number of rows in the CSV.

**What it does**
- Reads your CSV (expects columns for image paths and label paths).
- Renames files in-place (only the filenames change; directories remain the same).
- Preserves extensions.
- Logs a mapping (`old_path,new_path`) to `rename_mapping.csv` for record-keeping.
- Includes a **dry run** option to preview changes without modifying files.
- Provides a **rollback** cell that uses the mapping to revert changes.


In [1]:

# ==== Configuration ====
# Path to your CSV file:
CSV_PATH = "./8_band_outputs/paths.csv"   # e.g., '/home/you/data/paths.csv'

# Column names in your CSV:
IMAGE_COL = "image_path" # e.g., 'img', 'image', 'image_path'
LABEL_COL = "label_path" # e.g., 'mask', 'label', 'label_path'

# Naming parameters:
START_INDEX = 1          # starting index for numbering
IMAGE_PREFIX = "image"   # renaming prefix for images
LABEL_PREFIX = "image"   # renaming prefix for labels

# Safety & behavior:
DRY_RUN = False          # True = preview only, don't actually rename
SKIP_MISSING = True      # True = skip rows where files are missing; False = raise error
OVERWRITE_IF_EXISTS = False  # If a 'new' path already exists: False = error; True = overwrite

# Output mapping file (old_path,new_path)
MAPPING_CSV = "./8_band_outputs/rename_mapping.csv"


In [2]:

import os
import csv
from pathlib import Path
import pandas as pd

def _ensure_columns(df: pd.DataFrame, img_col: str, lbl_col: str):
    missing = [c for c in [img_col, lbl_col] if c not in df.columns]
    if missing:
        raise KeyError(f"CSV is missing required columns: {missing}. "
                       f"Available columns: {list(df.columns)}")

def _make_new_name(prefix: str, idx: int, old_path: Path) -> Path:
    # Keep same directory, change stem, preserve extension
    return old_path.with_name(f"{prefix}_{idx}{old_path.suffix}")

def _rename(old: Path, new: Path, overwrite: bool = False):
    if new.exists():
        if overwrite:
            # Remove existing target to allow rename
            if new.is_file():
                new.unlink()
            else:
                raise IsADirectoryError(f"Target exists and is a directory: {new}")
        else:
            raise FileExistsError(f"Target already exists: {new}")
    old.rename(new)

def _safe_exists(p: Path) -> bool:
    try:
        return p.exists()
    except Exception:
        return False


In [3]:

# ==== Execute Renaming ====

# Load CSV
df = pd.read_csv(CSV_PATH)
_ensure_columns(df, IMAGE_COL, LABEL_COL)

# Prepare logs
mappings = []  # each item: {'type': 'image'/'label', 'old_path': str, 'new_path': str, 'status': 'ok'/'skipped'/'error', 'reason': str}

count_total = 0
count_renamed = 0
count_skipped = 0
count_errors = 0

for i, row in df.iterrows():
    idx = START_INDEX + i
    img_old = Path(str(row[IMAGE_COL])).expanduser().resolve()
    lbl_old = Path(str(row[LABEL_COL])).expanduser().resolve()

    # Compute new targets (same dirs, new basenames)
    img_new = _make_new_name(IMAGE_PREFIX, idx, img_old)
    lbl_new = _make_new_name(LABEL_PREFIX, idx, lbl_old)

    count_total += 2  # image + label per row

    # Process IMAGE
    if not _safe_exists(img_old):
        reason = "missing source"
        if SKIP_MISSING:
            mappings.append({'type':'image','old_path':str(img_old),'new_path':str(img_new),'status':'skipped','reason':reason})
            count_skipped += 1
        else:
            mappings.append({'type':'image','old_path':str(img_old),'new_path':str(img_new),'status':'error','reason':reason})
            count_errors += 1
            raise FileNotFoundError(f"Image file not found: {img_old}")
    else:
        try:
            if DRY_RUN:
                mappings.append({'type':'image','old_path':str(img_old),'new_path':str(img_new),'status':'ok','reason':'dry-run'})
            else:
                _rename(img_old, img_new, overwrite=OVERWRITE_IF_EXISTS)
                mappings.append({'type':'image','old_path':str(img_old),'new_path':str(img_new),'status':'ok','reason':''})
            count_renamed += 1
        except Exception as e:
            mappings.append({'type':'image','old_path':str(img_old),'new_path':str(img_new),'status':'error','reason':str(e)})
            count_errors += 1

    # Process LABEL
    if not _safe_exists(lbl_old):
        reason = "missing source"
        if SKIP_MISSING:
            mappings.append({'type':'label','old_path':str(lbl_old),'new_path':str(lbl_new),'status':'skipped','reason':reason})
            count_skipped += 1
        else:
            mappings.append({'type':'label','old_path':str(lbl_old),'new_path':str(lbl_new),'status':'error','reason':reason})
            count_errors += 1
            raise FileNotFoundError(f"Label file not found: {lbl_old}")
    else:
        try:
            if DRY_RUN:
                mappings.append({'type':'label','old_path':str(lbl_old),'new_path':str(lbl_new),'status':'ok','reason':'dry-run'})
            else:
                _rename(lbl_old, lbl_new, overwrite=OVERWRITE_IF_EXISTS)
                mappings.append({'type':'label','old_path':str(lbl_old),'new_path':str(lbl_new),'status':'ok','reason':''})
            count_renamed += 1
        except Exception as e:
            mappings.append({'type':'label','old_path':str(lbl_old),'new_path':str(lbl_new),'status':'error','reason':str(e)})
            count_errors += 1

# Save mapping CSV
map_df = pd.DataFrame(mappings)
map_df.to_csv(MAPPING_CSV, index=False)

print("=== Summary ===")
print(f"Total files considered (img+label): {count_total}")
print(f"Renamed (or would rename in dry-run): {count_renamed}")
print(f"Skipped: {count_skipped}")
print(f"Errors: {count_errors}")
print(f"Mapping written to: {Path(MAPPING_CSV).resolve()}")
map_df.head(10)


=== Summary ===
Total files considered (img+label): 1600
Renamed (or would rename in dry-run): 1600
Skipped: 0
Errors: 0
Mapping written to: D:\Thesis\glacial_lake_detection_thesis\Training\3 Preparing train data\8_band_outputs\rename_mapping.csv


,type,old_path,new_path,status,reason
0,image,D:\Thesis\glacial_lake_detection_thesis\Traini...,D:\Thesis\glacial_lake_detection_thesis\Traini...,ok,
1,label,D:\Thesis\glacial_lake_detection_thesis\Traini...,D:\Thesis\glacial_lake_detection_thesis\Traini...,ok,
2,image,D:\Thesis\glacial_lake_detection_thesis\Traini...,D:\Thesis\glacial_lake_detection_thesis\Traini...,ok,
3,label,D:\Thesis\glacial_lake_detection_thesis\Traini...,D:\Thesis\glacial_lake_detection_thesis\Traini...,ok,
4,image,D:\Thesis\glacial_lake_detection_thesis\Traini...,D:\Thesis\glacial_lake_detection_thesis\Traini...,ok,
5,label,D:\Thesis\glacial_lake_detection_thesis\Traini...,D:\Thesis\glacial_lake_detection_thesis\Traini...,ok,
6,image,D:\Thesis\glacial_lake_detection_thesis\Traini...,D:\Thesis\glacial_lake_detection_thesis\Traini...,ok,
7,label,D:\Thesis\glacial_lake_detection_thesis\Traini...,D:\Thesis\glacial_lake_detection_thesis\Traini...,ok,
8,image,D:\Thesis\glacial_lake_detection_thesis\Traini...,D:\Thesis\glacial_lake_detection_thesis\Traini...,ok,
9,label,D:\Thesis\glacial_lake_detection_thesis\Traini...,D:\Thesis\glacial_lake_detection_thesis\Traini...,ok,


In [ ]:

from caas_jupyter_tools import display_dataframe_to_user
display_dataframe_to_user("Rename mapping preview", pd.read_csv(MAPPING_CSV))



## Rollback (Optional)

If you need to revert the renames, this cell reads `rename_mapping.csv` and swaps `new_path` → `old_path`.


In [ ]:

# ==== Rollback using rename_mapping.csv ====
# Set to True to perform rollback
DO_ROLLBACK = False

if DO_ROLLBACK:
    mapping = pd.read_csv(MAPPING_CSV)
    # Only consider rows with status 'ok' (successful renames or dry-run marked ok)
    ok_rows = mapping[mapping['status'] == 'ok'].copy()

    # We should roll back labels first and images after (or vice versa). Order typically doesn't matter,
    # but to be safe, we do reverse order of operations to avoid collisions.
    for _, r in ok_rows.iloc[::-1].iterrows():
        new_p = Path(str(r['new_path']))
        old_p = Path(str(r['old_path']))
        if new_p.exists():
            try:
                _rename(new_p, old_p, overwrite=OVERWRITE_IF_EXISTS)
            except Exception as e:
                print(f"Rollback error: {new_p} -> {old_p}: {e}")
        else:
            print(f"Rollback skip (missing current): {new_p}")
    print("Rollback complete.")
else:
    print("Rollback not executed. Set DO_ROLLBACK=True to run.")
